# **Stock Price Prediction Using LSTM Neural Network**

**Import Libraries**

In [16]:
import numpy as np
import pandas as pd
import yfinance as yf                 # Downloads Stock Market data from yahoo finance
import datetime
from datetime import date, timedelta

**Download and Preprocess the Dataset**

In [5]:
today = date.today()                   # Get today's date
d1 = today.strftime('%Y-%m-%d')        # Convert the date into string with given format
end_date = d1

d2 = date.today() - timedelta(days=5000)    # Calculate 5000 days earlier (approx 13-14 years)
d2 = d2.strftime('%Y-%m-%d')
start_date = d2

data = yf.download('AAPL',                   # Downloads original stock prices
                   end=end_date,
                   start=start_date,
                   progress=False)           # hides the pregress bar

data.columns = data.columns.get_level_values(0)          # Convert multi-index columns to normal columns

data['Date'] = data.index                                  # Convert Dataframe index into a normal Date column
data = data[['Date','Open','High','Low','Close','Volume']]       # Keep required columns
data.reset_index(drop=True, inplace=True)                 # Reset index to simple numbers (1,2,3,...)

/tmp/ipykernel_2241/3170772362.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('AAPL',


In [6]:
data.tail()

Price,Date,Open,High,Low,Close,Volume
3435,2026-06-24,295.359985,299.700012,292.940002,293.079987,53081900
3436,2026-06-25,287.399994,288.799988,273.750000,275.149994,107013700
3437,2026-06-26,275.000000,285.950012,274.209991,283.779999,261775500
3438,2026-06-29,286.730011,288.369995,279.850006,281.739990,66427000
3439,2026-06-30,281.170013,289.940002,280.700012,289.359985,65043800


**Visualization**

In [7]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data['Date'],     # X-axis - Trading dates
                                        open=data['Open'],
                                        high=data['High'],
                                        low=data['Low'],
                                        close=data['Close']
                                        )])
figure.update_layout(title='Apple Stock Price Analysis',
                     xaxis_rangeslider_visible=False)
figure.show()

**Correlation Analysis**

In [8]:
correlation = data.corr()           # Calculate corelation between all numerical columns
print(correlation['Close'].sort_values(ascending=False))    # Display how strongly each feature is correlated to 'Close Price'

Price
Close     1.000000
Low       0.999888
High      0.999885
Open      0.999749
Date      0.941998
Volume   -0.539622
Name: Close, dtype: float64


**Selecting Features (Input & Target)**

In [9]:
x = data[['Open','High','Low','Volume']]      # Input features (Variables)
y = data['Close']                             # Target feature (Variable)

x = x.to_numpy()             # Convert Dataframes into numpy arrays
y = y.to_numpy()             # Convert series into numpy array
y = y.reshape(-1, 1)         # Convert y into column vector


**Model Building**

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)    # 80% Training & 20% Testing

In [13]:
from keras.models import Sequential
from keras.layers import Dense, LSTM
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], 1)))       # LSTM layer with 128 memory units, and pass to the next LSTM layer
model.add(LSTM(64, return_sequences=False))                                      # Second LSTM layer with 64 memory units, output the final learned representation
model.add(Dense(25))                                     # Fully connected hidden layer
model.add(Dense(1))                                      # Output Layer
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 4, 128)         │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 25)             │         1,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,619 (459.45 KB)

 Trainable params: 117,619 (459.45 KB)

 Non-trainable params: 0 (0.00 B)

**Model Training**

In [14]:
model.compile(optimizer='adam',         # adam - Adjuts weights during learning
              loss='mean_squared_error')       # Error function to minimize
model.fit(x_train, y_train, batch_size=1, epochs=30)   # Train the model with repeating 30 times

Epoch 1/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 24s 8ms/step - loss: 1135.6982
Epoch 2/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - loss: 45.2885
Epoch 3/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 51.7854
Epoch 4/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 33.0509
Epoch 5/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 31.2602
Epoch 6/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 19.0519
Epoch 7/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 23.3661
Epoch 8/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 23.4425
Epoch 9/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 21.5302
Epoch 10/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 21.2143
Epoch 11/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 20.4533
Epoch 12/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 25.2356
Epoch 13/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 18.4856
Epoch 14/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 18.4520

**Prediction**

In [18]:
# Predict a New Stock Price (Format: [Open, High, Low, Volume])
new_day = np.array([[177.08,180.41,177.07,74919600]])     # Insert New Stock info
new_day = new_day.reshape((1,4,1))          # reshape the data into the format expected by the LSTM
predicted_Close = model.predict(new_day)

print(f'Predicted Closing Price: ${predicted_Close[0][0]:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Predicted Closing Price: $175.03


**Give it a Try yourslef!!!**

In [20]:
new_day2 = np.array([[]])      # (Format: [Open, High, Low, Volume])
new_day2 = new_day2.reshape((1,4,1))
predicted_close2 = model.predict(new_day2)

print(f'Predicted Closing Price: ${predicted_close2[0][0]:.2f}')

# **Project Summary**

## **Objective**

The objective of this project is to predict the **closing price of Apple (AAPL) stock** using historical stock market data. A **Long Short-Term Memory (LSTM)** neural network is employed to learn the relationship between selected stock market features and the corresponding closing price.

---

## **Dataset**

Historical Apple stock data was downloaded from **Yahoo Finance** using the `yfinance` library. Approximately **5000 days** of stock market data were collected.

The dataset consists of the following features:

| Feature | Description |
|----------|-------------|
| **Date** | Trading date |
| **Open** | Opening stock price |
| **High** | Highest stock price of the day |
| **Low** | Lowest stock price of the day |
| **Close** | Closing stock price (Target Variable) |
| **Volume** | Number of shares traded during the day |

---

The required libraries were imported to perform data collection, preprocessing, visualization, and model development. Historical Apple stock prices were downloaded for the specified date range, and the dataset was cleaned by flattening the column structure returned by the latest version of `yfinance`. The date index was converted into a separate **Date** column, only the required columns were retained, and the dataset index was reset for easier analysis.

An interactive **Candlestick Chart** was created using Plotly to visualize Apple's daily stock price movements. Each candlestick represents one trading day and displays the opening, highest, lowest, and closing prices, allowing the overall market trend and daily price fluctuations to be observed.

A correlation matrix was then computed to examine the relationship between the numerical features. The analysis showed that the **Open**, **High**, and **Low** prices have a strong positive correlation with the **Close** price, while **Volume** has a comparatively weaker relationship.

---

## **Model Development**

The input features selected for training were **Open**, **High**, **Low**, and **Volume**, while the **Close** price was used as the target variable. The dataset was converted into NumPy arrays and divided into **80% training data** and **20% testing data** using `train_test_split()`.

A **Long Short-Term Memory (LSTM)** neural network was built using Keras. The model consists of two LSTM layers followed by two Dense layers, enabling the network to learn complex relationships between the input features and the closing stock price. The model was compiled using the **Adam optimizer** and **Mean Squared Error (MSE)** as the loss function before being trained for **30 epochs**.

---

## **Prediction**

After training, the model learns the relationship between the selected stock market features and the closing price. New stock information containing the **Open**, **High**, **Low**, and **Volume** values can then be provided to the trained model to estimate the corresponding **closing stock price**.

---

## **Conclusion**

This project demonstrates the process of building a stock price prediction model using a **Long Short-Term Memory (LSTM)** neural network. Historical Apple stock data was collected, preprocessed, visualized, and analyzed before training the model. By learning patterns from the selected market features, the trained model is able to estimate the closing stock price for new input values, illustrating the application of deep learning techniques to financial data analysis.